# Milestone 4: Elevator outages in pipelines & PyTorch image classifier
*AI from scratch, milestone 4* · combine notebook for Stage 4

Build two complete real-world systems:
1. **Tabular pipeline**: predict long NYC subway elevator and escalator outages (>6 hours) from the live archive
   using pandas feature engineering and scikit-learn `Pipeline` + `ColumnTransformer`.
2. **Deep neural network**: train a PyTorch classifier with Batch Normalization, Dropout, and learning rate scheduling
   on image digits, diagnosing training curves.

Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)`
are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['df = raw_df[raw_df["finished"].astype(str) == "True"].copy().sort_values("start").reset_index(drop=True)', 'df["duration_hrs"] = (df["last_seen"] - df["start"]).dt.total_seconds() / 3600.0', 'df["is_long"] = (df["duration_hrs"] > 6.0).astype(int); df_outages = df'], 2: ['pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), X_cols)])', 'pipe = Pipeline([("pre", pre), ("model", HistGradientBoostingClassifier(random_state=42))]).fit(X_tr, y_tr)', 'auc_outages = float(roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1]))'], 3: ['digit_model = nn.Sequential(nn.Linear(64, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Linear(32, 10))'], 4: ['criterion = nn.CrossEntropyLoss(); opt = torch.optim.Adam(digit_model.parameters(), lr=0.01); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)', 'for ep in range(40): digit_model.train(); ... sched.step()', 'digit_model.eval(); with torch.no_grad(): torch_acc = float((digit_model(t_Xte).argmax(1) == t_yte).float().mean().item())']}
_ANSWERS = {1: 'ZGYgPSByYXdfZGZbcmF3X2RmWyJmaW5pc2hlZCJdLmFzdHlwZShzdHIpID09ICJUcnVlIl0uY29weSgpLnNvcnRfdmFsdWVzKCJzdGFydCIpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKZGZbImR1cmF0aW9uX2hycyJdID0gKGRmWyJsYXN0X3NlZW4iXSAtIGRmWyJzdGFydCJdKS5kdC50b3RhbF9zZWNvbmRzKCkgLyAzNjAwLjAKZGZbImlzX2xvbmciXSA9IChkZlsiZHVyYXRpb25faHJzIl0gPiA2LjApLmFzdHlwZShpbnQpCmRmX291dGFnZXMgPSBkZg==', 2: 'c3BsaXQgPSBpbnQobGVuKGRmX291dGFnZXMpICogMC43NSkKWF90ciwgeV90ciA9IGRmX291dGFnZXMuaWxvY1s6c3BsaXRdW1hfY29sc10sIGRmX291dGFnZXMuaWxvY1s6c3BsaXRdWyJpc19sb25nIl0KWF90ZSwgeV90ZSA9IGRmX291dGFnZXMuaWxvY1tzcGxpdDpdW1hfY29sc10sIGRmX291dGFnZXMuaWxvY1tzcGxpdDpdWyJpc19sb25nIl0KCnByZSA9IENvbHVtblRyYW5zZm9ybWVyKFsKICAgICgiY2F0IiwgT25lSG90RW5jb2RlcihoYW5kbGVfdW5rbm93bj0iaWdub3JlIiwgc3BhcnNlX291dHB1dD1GYWxzZSksIFhfY29scykKXSkKcGlwZSA9IFBpcGVsaW5lKFsKICAgICgicHJlIiwgcHJlKSwKICAgICgibW9kZWwiLCBIaXN0R3JhZGllbnRCb29zdGluZ0NsYXNzaWZpZXIocmFuZG9tX3N0YXRlPTQyKSkKXSkKcGlwZS5maXQoWF90ciwgeV90cikKcHJvYnMgPSBwaXBlLnByZWRpY3RfcHJvYmEoWF90ZSlbOiwgMV0KYXVjX291dGFnZXMgPSBmbG9hdChyb2NfYXVjX3Njb3JlKHlfdGUsIHByb2JzKSk=', 3: 'ZGlnaXRfbW9kZWwgPSBubi5TZXF1ZW50aWFsKAogICAgbm4uTGluZWFyKDY0LCA2NCksCiAgICBubi5CYXRjaE5vcm0xZCg2NCksCiAgICBubi5SZUxVKCksCiAgICBubi5Ecm9wb3V0KDAuMiksCiAgICBubi5MaW5lYXIoNjQsIDMyKSwKICAgIG5uLkJhdGNoTm9ybTFkKDMyKSwKICAgIG5uLlJlTFUoKSwKICAgIG5uLkxpbmVhcigzMiwgMTApCik=', 4: 'Y3JpdGVyaW9uID0gbm4uQ3Jvc3NFbnRyb3B5TG9zcygpCm9wdCA9IHRvcmNoLm9wdGltLkFkYW0oZGlnaXRfbW9kZWwucGFyYW1ldGVycygpLCBscj0wLjAxKQpzY2hlZCA9IHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5Db3NpbmVBbm5lYWxpbmdMUihvcHQsIFRfbWF4PTQwKQoKZm9yIGVwb2NoIGluIHJhbmdlKDQwKToKICAgIGRpZ2l0X21vZGVsLnRyYWluKCkKICAgIGZvciBieCwgYnkgaW4gbG9hZGVyOgogICAgICAgIG9wdC56ZXJvX2dyYWQoKQogICAgICAgIGxvc3MgPSBjcml0ZXJpb24oZGlnaXRfbW9kZWwoYngpLCBieSkKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICBvcHQuc3RlcCgpCiAgICBzY2hlZC5zdGVwKCkKCmRpZ2l0X21vZGVsLmV2YWwoKQp3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgIHByZWRzID0gZGlnaXRfbW9kZWwodF9YdGUpLmFyZ21heChkaW09MSkKICAgIHRvcmNoX2FjYyA9IGZsb2F0KChwcmVkcyA9PSB0X3l0ZSkuZmxvYXQoKS5tZWFuKCkuaXRlbSgpKQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import io, urllib.request
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.datasets import load_digits
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

## 1. Loading and filtering outages
Load the outage archive from the course data repository.
Filter for finished incidents (`finished == True` or `'True'`), sort chronologically by `start`,
and calculate `duration_hrs = (last_seen - start).total_seconds() / 3600.0`.
Create binary classification target: `is_long = (duration_hrs > 6.0).astype(int)`.
Assign the cleaned DataFrame to `df_outages`.

In [ ]:
outages_url = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/outages.csv"
raw_csv = urllib.request.urlopen(outages_url).read().decode()
raw_df = pd.read_csv(io.StringIO(raw_csv), parse_dates=["start", "last_seen"])

# YOUR CODE HERE (stuck? hint(1), then answer(1))
...

if "df_outages" in locals() and df_outages is not None:
    print(f"Cleaned completed outages: {len(df_outages)} rows. Long outages (>6h): {df_outages['is_long'].mean() * 100:.1f}%")

In [ ]:
check(1, lambda: len(df_outages) > 500 and "duration_hrs" in df_outages.columns and "is_long" in df_outages.columns, "outages table is filtered to completed incidents with duration in hours and binary target is_long")

## 2. Scikit-learn Pipeline with ColumnTransformer
Split the dataset chronologically: first 75% for training, remaining 25% for testing.
Features: `X_cols = ["kind", "reason"]`. Target: `"is_long"`.
Build a `Pipeline` with:
- `ColumnTransformer` with `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` on `X_cols`.
- `HistGradientBoostingClassifier(random_state=42)`.
Fit on the training set and compute validation ROC-AUC, saving to `auc_outages`.

In [ ]:
X_cols = ["kind", "reason"]

# YOUR CODE HERE (stuck? hint(2), then answer(2))
...

if "auc_outages" in locals() and auc_outages is not None:
    print(f"Outages Pipeline Test ROC-AUC: {auc_outages:.3f}")

In [ ]:
check(2, lambda: auc_outages > 0.65, "Scikit-learn pipeline encodes categories cleanly and achieves solid test AUC without data leakage")

## 3. PyTorch image network: Batch Normalization & Dropout
Now build a deep classifier in PyTorch on 8x8 handwritten digits.
Build an `nn.Sequential` named `digit_model`:
- `Linear(64, 64)`, `BatchNorm1d(64)`, `ReLU()`, `Dropout(0.2)`
- `Linear(64, 32)`, `BatchNorm1d(32)`, `ReLU()`
- `Linear(32, 10)`

In [ ]:
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...

In [ ]:
check(3, lambda: isinstance(digit_model, nn.Sequential) and any(isinstance(l, nn.BatchNorm1d) for l in digit_model) and any(isinstance(l, nn.Dropout) for l in digit_model), "digit_model architecture incorporates BatchNorm1d and Dropout layers")

## 4. Training PyTorch model with minibatches and scheduler
Prepare DataLoader with batch size 64.
Train `digit_model` using:
- `nn.CrossEntropyLoss()`
- `torch.optim.Adam(digit_model.parameters(), lr=0.01)`
- `torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)`
Train for 40 epochs. In test evaluation, call `digit_model.eval()` to disable Dropout.
Store the test accuracy on `X_img_te` in `torch_acc`.

In [ ]:
X_img, y_img = load_digits(return_X_y=True)
X_img = X_img / 16.0
n_tr = 1400
t_Xtr = torch.tensor(X_img[:n_tr], dtype=torch.float32)
t_ytr = torch.tensor(y_img[:n_tr], dtype=torch.int64)
t_Xte = torch.tensor(X_img[n_tr:], dtype=torch.float32)
t_yte = torch.tensor(y_img[n_tr:], dtype=torch.int64)

loader = DataLoader(TensorDataset(t_Xtr, t_ytr), batch_size=64, shuffle=True)

# YOUR CODE HERE (stuck? hint(4), then answer(4))
...

if "torch_acc" in locals() and torch_acc is not None:
    print(f"PyTorch image classifier test accuracy: {torch_acc * 100:.1f}%")

In [ ]:
check(4, lambda: torch_acc > 0.92, "Deep PyTorch image network reaches > 92% accuracy with batch normalization and cosine annealing")